In [1]:
import sys
import os

# Add src to path
sys.path.append(os.path.abspath('../src'))

# Database connection
from data.database import get_car_items

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import make_pipeline
# Add these imports to your existing ones
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor, make_column_transformer
from sklearn.metrics import make_scorer, mean_absolute_percentage_error
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import numpy as np
import pandas as pd


In [2]:
full_raw_dataset = get_car_items()
full_raw_dataset = full_raw_dataset.dropna(subset=['price', 'mileage', 'age_in_days'])

# print the top 10 most frequent models
print(full_raw_dataset['model'].value_counts().head(10))

model
DACIA_Spring      998
HYUNDAI_Kona      928
RENAULT_Megane    770
PEUGEOT_208       751
FIAT_500          748
RENAULT_Zoe       740
KIA_Niro          734
PEUGEOT_2008      676
NISSAN_Leaf       671
BMW_i3            399
Name: count, dtype: int64


In [3]:
SELECTED_MODEL = 'HYUNDAI_Kona'
single_model_dataset = full_raw_dataset[full_raw_dataset['model'] == SELECTED_MODEL]
print(f"Analyzing {SELECTED_MODEL} dataset: {single_model_dataset.shape[0]} samples")

Analyzing HYUNDAI_Kona dataset: 928 samples


In [4]:
# Your existing data preparation (KEEP AS IS)
features = ['mileage', 'age_in_days']
X, y = single_model_dataset[features], single_model_dataset['price']

# Split ONCE into train/test (80/20 is standard)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,  # Explicit test size
    random_state=0
)

In [5]:
def fit_model_grid_search(X_train, y_train, model):
    scoring = {
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error', 
        'mape': make_scorer(mean_absolute_percentage_error, greater_is_better=False),
        'rmse': 'neg_root_mean_squared_error'
    }
    grid_search = GridSearchCV(
        model['pipe'],
        model['param_grid'],
        cv=5,  # Internal cross-validation on training data
        scoring=scoring,
        refit='mape'
    )
    grid_search.fit(X_train, y_train)
    model['grid_search'] = grid_search
    results = grid_search.cv_results_
    model['best_model'] = grid_search.best_estimator_
    # CV metrics
    model['cv_r2'] = results['mean_test_r2'][grid_search.best_index_]
    model['cv_mae'] = -results['mean_test_mae'][grid_search.best_index_]
    model['cv_mape'] = -results['mean_test_mape'][grid_search.best_index_]
    model['cv_rmse'] = -results['mean_test_rmse'][grid_search.best_index_]

def model_evaluation(model, X_test=X_test, y_test=y_test):
    print("Best parameters found: ", model['grid_search'].best_params_)
    print("Cross-Validation Scores (training):")
    print(f"  CV R²:   {model['cv_r2']:.4f}")
    print(f"  CV MAE:  {model['cv_mae']:.0f}€")
    print(f"  CV MAPE: {model['cv_mape']:.2%}")
    print(f"  CV RMSE: {model['cv_rmse']:.0f}€")
    
    # Calculate ALL metrics on test set
    y_pred_test = model['best_model'].predict(X_test)
    model['test_r2'] = r2_score(y_test, y_pred_test)
    model['test_mae'] = mean_absolute_error(y_test, y_pred_test)
    model['test_mape'] = mean_absolute_percentage_error(y_test, y_pred_test)
    model['test_rmse'] = np.sqrt(mean_squared_error(y_test, y_pred_test))
    print("\nTest Set Scores (final evaluation):")
    print(f"  Test R²:   {model['test_r2']:.4f}")
    print(f"  Test MAE:  {model['test_mae']:.0f}€")
    print(f"  Test MAPE: {model['test_mape']:.2%}")
    print(f"  Test RMSE: {model['test_rmse']:.0f}€")
    # Access the best pipeline
    best_model = model['best_model']
    if not isinstance(best_model, VotingRegressor):
        # Feature information after preprocessing
        feature_names = best_model.named_steps['poly'].get_feature_names_out(['mileage', 'age_in_days'])
        print(f"Features after polynomial transformation: {feature_names}")

        # Model coefficients (for LinearRegression)
        regressor = best_model.named_steps['regressor']
        if isinstance(regressor, TransformedTargetRegressor):
            regressor = regressor.regressor_
        model['regressor'] = regressor
        print(f"\nModel coefficients: {regressor.coef_}")
        print(f"Model intercept: {regressor.intercept_}")

    sample_df = pd.DataFrame({
        'mileage': np.linspace(0, 50000, 6),
        'age_in_days': np.linspace(0, 365*5, 6)
    })
    print("\nSample predictions:")
    for i in range(len(sample_df)):
        print("Predicted Price for year={}, mileage={}: {:.0f}€".format(
            sample_df['age_in_days'].iloc[i] // 365,
            sample_df['mileage'].iloc[i],
            best_model.predict(sample_df.iloc[[i]])[0]
        ))

In [6]:
# Create pipeline with feature engineering
linear_regression = {
    "pipe": Pipeline([
        ('log_features', 'passthrough'),
        ('poly', PolynomialFeatures()),
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression())
    ]),
    "param_grid": {
        'log_features': ['passthrough', FunctionTransformer(np.log1p, inverse_func=np.expm1)],
        'poly__degree': [1, 2],
        'poly__interaction_only': [True, False]
    }
}

# Fit model using grid search
fit_model_grid_search(X_train, y_train, linear_regression)
model_evaluation(linear_regression)

Best parameters found:  {'log_features': 'passthrough', 'poly__degree': 2, 'poly__interaction_only': False}
Cross-Validation Scores (training):
  CV R²:   0.6176
  CV MAE:  2628€
  CV MAPE: 22.76%
  CV RMSE: 3366€

Test Set Scores (final evaluation):
  Test R²:   0.6951
  Test MAE:  2669€
  Test MAPE: 12.28%
  Test RMSE: 3444€
Features after polynomial transformation: ['1' 'mileage' 'age_in_days' 'mileage^2' 'mileage age_in_days'
 'age_in_days^2']

Model coefficients: [    0.         -1701.97519616 -7608.47787996   968.69343562
  -224.78762185  4622.8399533 ]
Model intercept: 21250.575471698114

Sample predictions:
Predicted Price for year=0.0, mileage=0.0: 37915€
Predicted Price for year=1.0, mileage=10000.0: 31137€
Predicted Price for year=2.0, mileage=20000.0: 25712€
Predicted Price for year=3.0, mileage=30000.0: 21642€
Predicted Price for year=4.0, mileage=40000.0: 18926€
Predicted Price for year=5.0, mileage=50000.0: 17563€


In [7]:
# Create pipeline with feature engineering
exponential_model_regression = {
    "pipe": Pipeline([
        ('log_features', 'passthrough'),
        ('negate', FunctionTransformer(lambda X: -X, inverse_func=lambda X: -X)),
        ('poly', PolynomialFeatures()),
        ('scaler', StandardScaler()),
        ('regressor', TransformedTargetRegressor(
            regressor=LinearRegression(),
            func=np.log1p,           # Transform target: x: log(x+1)
            inverse_func=np.expm1    # Inverse transform for predictions: x: exp(x)-1
        ))
    ]),
    "param_grid": {
        'log_features': ['passthrough', FunctionTransformer(np.log1p, inverse_func=np.expm1)],
        'poly__degree': [1, 2],
        'poly__interaction_only': [True, False]
    }
}

fit_model_grid_search(X_train, y_train, exponential_model_regression)
model_evaluation(exponential_model_regression)

Best parameters found:  {'log_features': 'passthrough', 'poly__degree': 2, 'poly__interaction_only': False}
Cross-Validation Scores (training):
  CV R²:   0.6154
  CV MAE:  2610€
  CV MAPE: 22.41%
  CV RMSE: 3378€

Test Set Scores (final evaluation):
  Test R²:   0.6915
  Test MAE:  2663€
  Test MAPE: 12.06%
  Test RMSE: 3464€
Features after polynomial transformation: ['1' 'mileage' 'age_in_days' 'mileage^2' 'mileage age_in_days'
 'age_in_days^2']

Model coefficients: [ 0.          0.073273    0.2682514   0.05134234 -0.02682557  0.15107755]
Model intercept: 9.931419833289025

Sample predictions:
Predicted Price for year=0.0, mileage=0.0: 39129€
Predicted Price for year=1.0, mileage=10000.0: 30543€
Predicted Price for year=2.0, mileage=20000.0: 24871€
Predicted Price for year=3.0, mileage=30000.0: 21127€
Predicted Price for year=4.0, mileage=40000.0: 18723€
Predicted Price for year=5.0, mileage=50000.0: 17308€


In [8]:
# Ultra-concise version
mixed_model = {
    'pipe': VotingRegressor([
        ('linear', linear_regression['best_model']),
        ('exponential', exponential_model_regression['best_model'])
    ]),
    'param_grid': {
        'weights': [[a, 1-a] for a in np.linspace(0, 1, 11)]
    }
}

fit_model_grid_search(X_train, y_train, mixed_model)
model_evaluation(mixed_model)

Best parameters found:  {'weights': [np.float64(0.0), np.float64(1.0)]}
Cross-Validation Scores (training):
  CV R²:   0.6154
  CV MAE:  2610€
  CV MAPE: 22.41%
  CV RMSE: 3378€

Test Set Scores (final evaluation):
  Test R²:   0.6915
  Test MAE:  2663€
  Test MAPE: 12.06%
  Test RMSE: 3464€

Sample predictions:
Predicted Price for year=0.0, mileage=0.0: 39129€
Predicted Price for year=1.0, mileage=10000.0: 30543€
Predicted Price for year=2.0, mileage=20000.0: 24871€
Predicted Price for year=3.0, mileage=30000.0: 21127€
Predicted Price for year=4.0, mileage=40000.0: 18723€
Predicted Price for year=5.0, mileage=50000.0: 17308€
